# 18.5 端侧与移动端部署 (Edge & Mobile Deployment)

> 🕐 预估学习时间：35分钟

本节介绍大语言模型在端侧与移动端的部署技术。随着模型压缩与硬件加速的发展，将小型 LLM 部署到手机、笔记本和 IoT 设备成为可能，从而实现隐私保护、低延迟和离线推理。

**学习主题：**

1. 端侧部署概述与设备能力分析（DeviceProfile 类）
2. 量化与格式转换（QuantizationConverter 类）
3. 模型裁剪与蒸馏（EdgeOptimizer 类）
4. 推理引擎对比（EdgeInferenceEngine 类）
5. 端云协同（EdgeCloudRouter 类）

## 1. 端侧部署概述

端侧部署指将模型直接运行在用户设备上，而非云端服务器。这种方式在隐私、延迟和成本方面具有独特优势，但也面临内存、算力和电池等约束。

**端侧部署的核心动机：**

- **隐私保护**：用户数据不出设备，满足 GDPR、HIPAA 等合规要求
- **低延迟**：无需网络往返，交互式应用响应更快
- **成本节约**：减少云端 GPU 服务器和带宽开销
- **离线可用**：在网络不可用场景下仍能提供服务

**主要挑战：**

- **内存限制**：手机通常只有 2-4GB 可用内存给模型
- **算力受限**：移动 CPU/GPU 算力远低于服务器
- **能耗约束**：持续推理会快速消耗电池
- **热设计**：长时间高负载会导致设备过热降频

**目标设备：**

- **手机**：iOS / Android，NPU + GPU + CPU 异构
- **笔记本**：Apple Silicon (M 系列) / x86 + 独立显卡
- **边缘设备**：Jetson、树莓派、专用 AI 加速棒

In [ ]:
import torch
import torch.nn as nn
import math
from collections import OrderedDict

torch.manual_seed(42)

print('=== 端侧设备能力分析 ===')

class DeviceProfile:
    '''设备规格画像：描述端侧设备的硬件能力'''

    def __init__(self, name, ram_gb, compute_tops, battery_wh, npu=False):
        self.name = name
        self.ram_gb = ram_gb
        self.compute_tops = compute_tops
        self.battery_wh = battery_wh
        self.npu = npu

    def available_model_memory_gb(self, fraction=0.4):
        '''模型可用内存（设备内存的一部分）'''
        return self.ram_gb * fraction

    def power_budget_w(self, sustained_fraction=0.5):
        '''持续推理功率预算（瓦特）'''
        return self.compute_tops * sustained_fraction

    def describe(self):
        npu_text = '有' if self.npu else '无'
        return OrderedDict([
            ('设备', self.name),
            ('内存(GB)', self.ram_gb),
            ('算力(TOPS)', self.compute_tops),
            ('电池(Wh)', self.battery_wh),
            ('NPU', npu_text),
            ('模型可用内存(GB)', round(self.available_model_memory_gb(), 2)),
        ])

class ModelRequirements:
    '''模型需求画像：估算模型在端侧的占用与推理时间'''

    def __init__(self, name, params_b, bits=16):
        self.name = name
        self.params_b = params_b
        self.bits = bits

    def size_gb(self):
        '''模型权重占用大小（GB）'''
        return self.params_b * self.bits / 8.0

    def runtime_memory_gb(self, overhead=1.3):
        '''运行时内存（含 KV cache 与激活）'''
        return self.size_gb() * overhead

    def estimate_tokens_per_sec(self, device_compute_tops, efficiency=0.15):
        '''估算每秒生成 token 数'''
        ops_per_token = self.params_b * 2 * 1e9
        effective_tops = device_compute_tops * 1e12 * efficiency
        return effective_tops / ops_per_token

    def describe(self):
        return OrderedDict([
            ('模型', self.name),
            ('参数量(B)', self.params_b),
            ('量化位数', self.bits),
            ('权重大小(GB)', round(self.size_gb(), 2)),
            ('运行时内存(GB)', round(self.runtime_memory_gb(), 2)),
        ])

# 定义设备
devices = [
    DeviceProfile('旗舰手机', ram_gb=8, compute_tops=15, battery_wh=15, npu=True),
    DeviceProfile('中端手机', ram_gb=6, compute_tops=8, battery_wh=12, npu=True),
    DeviceProfile('Apple M2 笔记本', ram_gb=16, compute_tops=30, battery_wh=60, npu=True),
    DeviceProfile('Jetson Orin 边缘盒', ram_gb=8, compute_tops=40, battery_wh=0, npu=True),
]

# 定义候选模型
models = [
    ModelRequirements('TinyLlama-1.1B', params_b=1.1, bits=4),
    ModelRequirements('Phi-3-mini-3.8B', params_b=3.8, bits=4),
    ModelRequirements('Qwen2-7B', params_b=7.0, bits=4),
    ModelRequirements('Llama-3-8B', params_b=8.0, bits=8),
]

print('\n--- 设备规格表 ---')
print(f"{'设备':<18}{'内存GB':>8}{'算力TOPS':>10}{'电池Wh':>8}{'NPU':>5}")
for d in devices:
    info = d.describe()
    npu_text = '有' if d.npu else '无'
    print(f"{d.name:<18}{d.ram_gb:>8}{d.compute_tops:>10}{d.battery_wh:>8}{npu_text:>5}")

print('\n--- 模型需求表 ---')
print(f"{'模型':<22}{'参数B':>7}{'位数':>5}{'权重GB':>9}{'运行GB':>9}")
for m in models:
    info = m.describe()
    print(f"{m.name:<22}{m.params_b:>7}{m.bits:>5}{m.size_gb():>9.2f}{m.runtime_memory_gb():>9.2f}")

print('\n--- 模型-设备兼容性矩阵 ---')
header = f"{'模型':<22}"
for d in devices:
    header += f"{d.name[:10]:>12}"
print(header)

for m in models:
    row = f"{m.name:<22}"
    for d in devices:
        mem_ok = m.runtime_memory_gb() <= d.available_model_memory_gb()
        tps = m.estimate_tokens_per_sec(d.compute_tops)
        speed_ok = tps >= 3.0
        if mem_ok and speed_ok:
            verdict = f'{tps:.1f}t/s'
        elif mem_ok:
            verdict = '慢'
        else:
            verdict = '不兼容'
        row += f"{verdict:>12}"
    print(row)

print(f'\nKey: 端侧部署需要同时满足内存约束（模型运行时内存 ≤ 设备可用内存）和算力约束（每秒 token 数 ≥ 交互阈值），通过 DeviceProfile 与 ModelRequirements 可快速筛选可行组合')

## 2. 量化与格式转换

量化是端侧部署最核心的技术，通过降低权重精度大幅压缩模型体积并加速推理。

**主流量化格式：**

- **GGUF**：llama.cpp 生态主流格式，支持 Q4_K_M、Q5_K_S 等混合精度量化
- **Q4_K_M**：4 比特为主，关键层保留更高精度，体积小质量好
- **Q5_K_S**：5 比特量化，质量更接近原模型，体积略大
- **ONNX**：跨平台格式，可在 ONNX Runtime 上运行，支持 INT8 量化
- **Core ML**：Apple 平台原生格式，针对 Neural Engine 优化
- **MLX**：Apple Silicon 原生框架，支持 4/8 比特量化

**量化质量-体积权衡：**

量化位数越低，体积越小、速度越快，但质量损失越大。Q4_K_M 通常是性价比最高的选择，在体积与质量之间取得平衡。

In [ ]:
import torch
import torch.nn as nn
import math
from collections import OrderedDict

torch.manual_seed(42)

print('=== 量化格式对比 QuantizationConverter ===')

class QuantizationConverter:
    '''模拟将模型转换为不同量化格式，评估体积/内存/质量权衡'''

    # 格式配置：位数、混合精度系数、质量保留率、运行时开销
    FORMATS = OrderedDict([
        ('FP16', {'bits': 16, 'size_factor': 1.0, 'quality': 1.000, 'runtime_overhead': 1.4}),
        ('INT8', {'bits': 8, 'size_factor': 1.0, 'quality': 0.985, 'runtime_overhead': 1.25}),
        ('Q4', {'bits': 4, 'size_factor': 1.0, 'quality': 0.940, 'runtime_overhead': 1.15}),
        ('Q4_K_M', {'bits': 4, 'size_factor': 1.08, 'quality': 0.965, 'runtime_overhead': 1.18}),
        ('Q5_K_S', {'bits': 5, 'size_factor': 1.0, 'quality': 0.978, 'runtime_overhead': 1.20}),
    ])

    def __init__(self, model_name, params_b):
        self.model_name = model_name
        self.params_b = params_b

    def convert(self, format_name):
        '''模拟转换到指定格式'''
        cfg = self.FORMATS[format_name]
        size_gb = self.params_b * cfg['bits'] / 8.0 * cfg['size_factor']
        runtime_gb = size_gb * cfg['runtime_overhead']
        # 模拟基线困惑度 5.5，量化后按质量保留率反推
        base_ppl = 5.5
        quantized_ppl = base_ppl / cfg['quality']
        ppl_increase = quantized_ppl - base_ppl
        return OrderedDict([
            ('format', format_name),
            ('bits', cfg['bits']),
            ('size_gb', round(size_gb, 2)),
            ('runtime_gb', round(runtime_gb, 2)),
            ('ppl', round(quantized_ppl, 3)),
            ('ppl_increase', round(ppl_increase, 3)),
            ('quality', cfg['quality']),
        ])

    def convert_all(self):
        '''转换到所有格式'''
        return {fmt: self.convert(fmt) for fmt in self.FORMATS}

# 对多个模型进行量化对比
models_to_test = [
    ('TinyLlama-1.1B', 1.1),
    ('Phi-3-mini-3.8B', 3.8),
    ('Llama-3-8B', 8.0),
]

for model_name, params_b in models_to_test:
    print(f'\n--- {model_name} ({params_b}B 参数) 量化对比 ---')
    converter = QuantizationConverter(model_name, params_b)
    results = converter.convert_all()

    print(f"{'格式':<10}{'位数':>5}{'权重GB':>9}{'运行GB':>9}{'困惑度':>8}{'PPL增加':>9}{'质量':>7}")
    for fmt, info in results.items():
        print(f"{fmt:<10}{info['bits']:>5}{info['size_gb']:>9.2f}{info['runtime_gb']:>9.2f}"
              f"{info['ppl']:>8.3f}{info['ppl_increase']:>9.3f}{info['quality']:>7.1%}")

# 量化格式推荐
print('\n--- 量化格式推荐 ---')
recommendations = OrderedDict([
    ('旗舰手机（8GB）', 'Q4_K_M（体积小、质量优、llama.cpp 原生支持）'),
    ('Apple Silicon', 'MLX 4bit（针对统一内存优化）或 Q4_K_M'),
    ('边缘服务器', 'INT8 ONNX（跨平台、易部署）或 Q5_K_S（质量更优）'),
    ('极致质量', 'FP16（无质量损失，但体积大）'),
])
for scenario, fmt in recommendations.items():
    print(f'  {scenario}: {fmt}')

print(f'\nKey: Q4_K_M 是端侧部署的甜点选择，体积约为 FP16 的 1/4，质量保留 96.5% 以上；Q5_K_S 在质量敏感场景下更优；FP16 仅用于资源充裕的边缘服务器')

## 3. 模型裁剪与蒸馏

除了量化，端侧部署还常通过裁剪与蒸馏进一步压缩模型，使其适配更小的设备。

**主要技术：**

- **任务特定裁剪**：移除与目标任务无关的层或参数
- **词表裁剪**：移除未使用的 token，减少 embedding 层体积
- **层丢弃（Layer Dropping）**：移除冗余层，保留关键层
- **注意力头裁剪**：移除贡献小的注意力头
- **知识蒸馏**：用大模型蒸馏出更小的端侧专用模型

**裁剪策略：**

层丢弃通常移除中间层（首尾层影响较大需保留），词表裁剪需根据目标语料统计 token 频率。裁剪后通常需要少量微调恢复质量。

In [ ]:
import torch
import torch.nn as nn
import math
from collections import OrderedDict

torch.manual_seed(42)

print('=== 端侧模型优化 EdgeOptimizer ===')

class EdgeOptimizer:
    '''端侧模型优化：词表裁剪、层丢弃、注意力头裁剪'''

    def __init__(self, model_name, vocab_size=32000, n_layers=32, n_heads=32, hidden=4096):
        self.model_name = model_name
        self.vocab_size = vocab_size
        self.n_layers = n_layers
        self.n_heads = n_heads
        self.hidden = hidden
        self.actions = []
        self._original_params = None

    def base_params_b(self):
        '''估算参数量（B）'''
        embed = self.vocab_size * self.hidden
        per_layer = 12 * self.hidden * self.hidden  # 近似 transformer 层参数
        total = embed + self.n_layers * per_layer
        return total / 1e9

    def base_size_mb(self, bits=16):
        '''基线模型大小（MB）'''
        return self.base_params_b() * 1e9 * bits / 8 / 1e6

    def prune_vocab(self, keep_ratio=0.3):
        '''词表裁剪：保留高频 token'''
        if self._original_params is None:
            self._original_params = self.base_params_b()
        old_vocab = self.vocab_size
        self.vocab_size = max(1000, int(old_vocab * keep_ratio))
        saved_ratio = 1 - (self.vocab_size / old_vocab)
        embed_share = (old_vocab * self.hidden) / (self._original_params * 1e9)
        size_reduction = saved_ratio * embed_share
        quality_drop = 0.01 * saved_ratio
        self.actions.append(OrderedDict([
            ('action', '词表裁剪'),
            ('old_vocab', old_vocab),
            ('new_vocab', self.vocab_size),
            ('size_reduction', size_reduction),
            ('quality_drop', quality_drop),
        ]))
        return size_reduction, quality_drop

    def drop_layers(self, keep_layers=20):
        '''层丢弃：保留前 N 层和后 M 层，移除中间冗余层'''
        if self._original_params is None:
            self._original_params = self.base_params_b()
        old_layers = self.n_layers
        self.n_layers = min(keep_layers, old_layers)
        dropped = old_layers - self.n_layers
        layer_share = (old_layers * 12 * self.hidden * self.hidden) / (self._original_params * 1e9)
        size_reduction = (dropped / old_layers) * layer_share
        quality_drop = 0.03 * (dropped / old_layers)
        self.actions.append(OrderedDict([
            ('action', '层丢弃'),
            ('old_layers', old_layers),
            ('new_layers', self.n_layers),
            ('dropped', dropped),
            ('size_reduction', size_reduction),
            ('quality_drop', quality_drop),
        ]))
        return size_reduction, quality_drop

    def prune_heads(self, keep_ratio=0.75):
        '''注意力头裁剪'''
        if self._original_params is None:
            self._original_params = self.base_params_b()
        old_heads = self.n_heads
        self.n_heads = max(4, int(old_heads * keep_ratio))
        pruned = old_heads - self.n_heads
        head_share = 0.33
        size_reduction = (pruned / old_heads) * head_share * 0.5
        quality_drop = 0.015 * (pruned / old_heads)
        self.actions.append(OrderedDict([
            ('action', '注意力头裁剪'),
            ('old_heads', old_heads),
            ('new_heads', self.n_heads),
            ('pruned', pruned),
            ('size_reduction', size_reduction),
            ('quality_drop', quality_drop),
        ]))
        return size_reduction, quality_drop

    def summary(self, bits=4):
        '''输出优化摘要'''
        original_params = self._original_params if self._original_params else self.base_params_b()
        current_params = self.base_params_b()
        original_size = original_params * 1e9 * bits / 8 / 1e6
        current_size = current_params * 1e9 * bits / 8 / 1e6
        total_reduction = 1 - (current_params / original_params)
        total_quality_drop = sum(a['quality_drop'] for a in self.actions)
        return OrderedDict([
            ('model', self.model_name),
            ('original_params_b', round(original_params, 3)),
            ('current_params_b', round(current_params, 3)),
            ('original_size_mb', round(original_size, 1)),
            ('current_size_mb', round(current_size, 1)),
            ('size_reduction', round(total_reduction, 3)),
            ('quality_drop', round(total_quality_drop, 3)),
            ('actions', len(self.actions)),
        ])

# 演示端侧模型优化
print('--- 优化前基线 ---')
optimizer = EdgeOptimizer('Llama-3-8B', vocab_size=128256, n_layers=32, n_heads=32, hidden=4096)
print(f'模型: {optimizer.model_name}')
print(f'基线参数量: {optimizer.base_params_b():.3f}B')
print(f'基线大小(FP16): {optimizer.base_size_mb(16):.1f}MB')
print(f'基线大小(Q4): {optimizer.base_size_mb(4):.1f}MB')

print('\n--- 应用优化策略 ---')
# 1. 词表裁剪：任务相关 token 保留 30%
r1, q1 = optimizer.prune_vocab(keep_ratio=0.3)
print(f'词表裁剪: 保留 30%，体积减少 {r1:.1%}，质量损失 {q1:.1%}')

# 2. 层丢弃：保留 20 层
r2, q2 = optimizer.drop_layers(keep_layers=20)
print(f'层丢弃: 保留 20 层，体积减少 {r2:.1%}，质量损失 {q2:.1%}')

# 3. 注意力头裁剪：保留 75%
r3, q3 = optimizer.prune_heads(keep_ratio=0.75)
print(f'注意力头裁剪: 保留 75%，体积减少 {r3:.1%}，质量损失 {q3:.1%}')

print('\n--- 优化摘要 ---')
summary = optimizer.summary(bits=4)
for k, v in summary.items():
    if k == 'actions':
        continue
    print(f'  {k}: {v}')

print('\n--- 优化动作详情 ---')
for i, action in enumerate(optimizer.actions, 1):
    print(f'  {i}. {action["action"]}: 体积减少 {action["size_reduction"]:.1%}, 质量损失 {action["quality_drop"]:.1%}')

print(f'\nKey: 端侧模型优化通过词表裁剪（任务相关 token）、层丢弃（移除冗余中间层）和注意力头裁剪（移除低贡献头）三步组合，可在量化基础上进一步压缩 30-50% 体积，但需配合少量微调以恢复质量')

## 4. 推理引擎对比

选择合适的推理引擎对端侧部署的性能至关重要，不同引擎针对不同硬件平台优化。

**主流端侧推理引擎：**

- **llama.cpp**：C++ 实现，支持 CPU/GPU，跨平台，GGUF 格式，社区活跃
- **MLC-LLM**：针对移动 GPU 优化，支持 iOS/Android，基于 Apache TVM
- **MLX**：Apple 官方框架，针对 Apple Silicon 统一内存优化
- **ONNX Runtime**：微软跨平台推理引擎，支持 INT8 量化，企业级部署
- **TensorRT Mobile**：NVIDIA 针对移动端优化的推理引擎

**引擎选择考量：**

- **平台兼容性**：iOS 优先 MLC/MLX，Android 优先 MLC/llama.cpp，跨平台选 ONNX Runtime
- **硬件加速**：是否利用 NPU/Neural Engine/Metal/Vulkan
- **内存效率**：统一内存架构（Apple Silicon）vs 独立显存
- **生态成熟度**：模型格式支持、社区支持、文档完善度

In [ ]:
import torch
import torch.nn as nn
import math
from collections import OrderedDict

torch.manual_seed(42)

print('=== 端侧推理引擎对比 EdgeInferenceEngine ===')

class EdgeInferenceEngine:
    '''模拟不同推理引擎在端侧设备上的性能'''

    ENGINES = OrderedDict([
        ('llama.cpp CPU', {'backend': 'CPU', 'mem_eff': 0.85, 'energy_eff': 0.6, 'init_ms': 800, 'speed_factor': 1.0}),
        ('llama.cpp GPU', {'backend': 'GPU', 'mem_eff': 0.75, 'energy_eff': 0.7, 'init_ms': 1200, 'speed_factor': 1.8}),
        ('MLC-LLM', {'backend': 'Mobile GPU', 'mem_eff': 0.80, 'energy_eff': 0.85, 'init_ms': 600, 'speed_factor': 2.0}),
        ('MLX', {'backend': 'Apple Silicon', 'mem_eff': 0.95, 'energy_eff': 0.9, 'init_ms': 400, 'speed_factor': 2.5}),
        ('ONNX Runtime', {'backend': 'CPU/NPU', 'mem_eff': 0.82, 'energy_eff': 0.75, 'init_ms': 500, 'speed_factor': 1.5}),
    ])

    def __init__(self, model_name, params_b, bits=4):
        self.model_name = model_name
        self.params_b = params_b
        self.bits = bits

    def model_size_gb(self):
        return self.params_b * self.bits / 8.0

    def benchmark(self, engine_name, device_compute_tops, device_battery_wh):
        '''模拟在指定设备上的推理性能'''
        cfg = self.ENGINES[engine_name]
        base_ops = self.params_b * 2 * 1e9
        effective_tops = device_compute_tops * 1e12 * 0.15
        base_tps = effective_tops / base_ops
        tps = base_tps * cfg['speed_factor']

        runtime_gb = self.model_size_gb() * 1.2 / cfg['mem_eff']

        power_w = device_compute_tops * 0.4 * (1 / cfg['energy_eff'])
        if device_battery_wh > 0:
            battery_hours = device_battery_wh / power_w
        else:
            battery_hours = -1.0

        ttft_ms = cfg['init_ms'] + 1000 / max(tps, 0.1) * 5

        return OrderedDict([
            ('engine', engine_name),
            ('backend', cfg['backend']),
            ('tokens_per_sec', round(tps, 2)),
            ('runtime_gb', round(runtime_gb, 2)),
            ('power_w', round(power_w, 2)),
            ('battery_hours', round(battery_hours, 1)),
            ('ttft_ms', round(ttft_ms, 0)),
        ])

    def benchmark_all(self, device_compute_tops, device_battery_wh):
        '''在指定设备上测试所有引擎'''
        return {name: self.benchmark(name, device_compute_tops, device_battery_wh)
                for name in self.ENGINES}

# 测试 Phi-3-mini-3.8B (Q4) 在不同设备上的表现
engine = EdgeInferenceEngine('Phi-3-mini-3.8B', params_b=3.8, bits=4)

test_devices = [
    ('旗舰手机', 15, 15),
    ('Apple M2 笔记本', 30, 60),
    ('Jetson Orin 边缘盒', 40, 0),
]

for device_name, compute, battery in test_devices:
    print(f'\n--- {device_name} (算力 {compute} TOPS, 电池 {battery} Wh) ---')
    results = engine.benchmark_all(compute, battery)
    print(f"{'引擎':<18}{'后端':<16}{'t/s':>7}{'内存GB':>8}{'功耗W':>8}{'续航h':>8}{'TTFTms':>8}")
    for name, info in results.items():
        battery_text = f"{info['battery_hours']:.1f}" if info['battery_hours'] > 0 else 'N/A'
        print(f"{name:<18}{info['backend']:<16}{info['tokens_per_sec']:>7.2f}"
              f"{info['runtime_gb']:>8.2f}{info['power_w']:>8.2f}{battery_text:>8}{info['ttft_ms']:>8.0f}")

# 引擎推荐
print('\n--- 引擎选择推荐 ---')
recommendations = OrderedDict([
    ('iOS (iPhone)', 'MLC-LLM（移动 GPU 加速、能效优）或 MLX（M 系列芯片）'),
    ('Android', 'MLC-LLM 或 llama.cpp GPU（Vulkan 后端）'),
    ('Apple Silicon', 'MLX（统一内存、Metal 优化）'),
    ('x86 笔记本', 'llama.cpp CPU/GPU 或 ONNX Runtime'),
    ('边缘服务器', 'ONNX Runtime（企业级）或 llama.cpp（社区生态）'),
])
for scenario, eng in recommendations.items():
    print(f'  {scenario}: {eng}')

print(f'\nKey: 推理引擎选择需匹配硬件平台：Apple Silicon 选 MLX（统一内存优势），移动端选 MLC-LLM（移动 GPU 优化），跨平台选 llama.cpp 或 ONNX Runtime；能效比直接决定电池续航')

## 5. 端云协同

端云协同（Edge-Cloud Collaboration）结合端侧的低延迟、隐私优势与云端的高质量、强算力优势，是实际部署的常见架构。

**主要模式：**

- **混合架构**：简单请求端侧处理，复杂请求转发云端
- **模型级联（Cascading）**：端侧小模型先尝试，置信度不足时调用云端大模型
- **推测式端侧解码**：端侧小模型快速生成草稿，云端大模型验证修正
- **云端蒸馏**：云端大模型持续蒸馏更新端侧小模型

**路由策略：**

基于置信度的路由是最常用策略：端侧模型输出置信度，低于阈值则转发云端。可在保证质量的同时大幅降低云端负载与延迟。

In [ ]:
import torch
import torch.nn as nn
import math
from collections import OrderedDict

torch.manual_seed(42)

print('=== 端云协同路由 EdgeCloudRouter ===')

class EdgeCloudRouter:
    '''端云协同路由：基于置信度路由 + 推测式端侧解码'''

    def __init__(self, confidence_threshold=0.85):
        self.threshold = confidence_threshold
        self.edge_latency_ms = 80
        self.cloud_latency_ms = 600
        self.network_latency_ms = 200
        self.routing_log = []

    def edge_model_predict(self, query_id, n_classes=5):
        '''模拟端侧小模型预测'''
        logits = torch.randn(n_classes) * 2
        probs = torch.softmax(logits, dim=0)
        confidence, pred = probs.max(dim=0)
        return pred.item(), confidence.item(), probs

    def cloud_model_predict(self, query_id, n_classes=5):
        '''模拟云端大模型预测（质量更高）'''
        logits = torch.randn(n_classes) * 3 + 1
        probs = torch.softmax(logits, dim=0)
        confidence, pred = probs.max(dim=0)
        return pred.item(), confidence.item(), probs

    def route(self, query_id):
        '''基于置信度路由单个请求'''
        edge_pred, edge_conf, edge_probs = self.edge_model_predict(query_id)

        if edge_conf >= self.threshold:
            latency = self.edge_latency_ms
            decision = 'edge'
            final_pred = edge_pred
            final_conf = edge_conf
        else:
            cloud_pred, cloud_conf, cloud_probs = self.cloud_model_predict(query_id)
            latency = self.edge_latency_ms + self.network_latency_ms + 100
            decision = 'cloud'
            final_pred = cloud_pred
            final_conf = cloud_conf

        record = OrderedDict([
            ('query_id', query_id),
            ('decision', decision),
            ('edge_conf', round(edge_conf, 3)),
            ('final_pred', final_pred),
            ('final_conf', round(final_conf, 3)),
            ('latency_ms', latency),
        ])
        self.routing_log.append(record)
        return record

    def speculative_edge_decode(self, query_id, draft_tokens=8):
        '''推测式端侧解码：端侧生成草稿，云端验证'''
        draft_latency = self.edge_latency_ms * 0.5
        draft = [torch.randint(0, 1000, (1,)).item() for _ in range(draft_tokens)]

        accept_rate = 0.7
        accepted = int(draft_tokens * accept_rate)
        rejected = draft_tokens - accepted

        verify_latency = self.network_latency_ms + 50
        regen_latency = rejected * (self.network_latency_ms / draft_tokens)
        total_latency = draft_latency + verify_latency + regen_latency

        pure_cloud_latency = draft_tokens * (self.network_latency_ms + 100) / draft_tokens * 2
        speedup = pure_cloud_latency / total_latency

        return OrderedDict([
            ('query_id', query_id),
            ('draft_tokens', draft_tokens),
            ('accepted', accepted),
            ('rejected', rejected),
            ('accept_rate', round(accept_rate, 2)),
            ('spec_latency_ms', round(total_latency, 0)),
            ('pure_cloud_latency_ms', round(pure_cloud_latency, 0)),
            ('speedup', round(speedup, 2)),
        ])

    def analyze(self):
        '''分析路由统计'''
        if not self.routing_log:
            return None
        total = len(self.routing_log)
        edge_count = sum(1 for r in self.routing_log if r['decision'] == 'edge')
        cloud_count = total - edge_count
        avg_latency = sum(r['latency_ms'] for r in self.routing_log) / total
        edge_avg_latency = (sum(r['latency_ms'] for r in self.routing_log if r['decision'] == 'edge')
                            / max(edge_count, 1))
        cloud_avg_latency = (sum(r['latency_ms'] for r in self.routing_log if r['decision'] == 'cloud')
                             / max(cloud_count, 1))
        return OrderedDict([
            ('total_requests', total),
            ('edge_handled', edge_count),
            ('cloud_handled', cloud_count),
            ('edge_ratio', round(edge_count / total, 2)),
            ('avg_latency_ms', round(avg_latency, 0)),
            ('edge_avg_latency_ms', round(edge_avg_latency, 0)),
            ('cloud_avg_latency_ms', round(cloud_avg_latency, 0)),
        ])

# 演示端云协同路由
router = EdgeCloudRouter(confidence_threshold=0.40)

print('--- 置信度路由决策 ---')
print(f"{'请求ID':>8}{'决策':>8}{'端侧置信':>10}{'最终预测':>10}{'最终置信':>10}{'延迟ms':>9}")
for qid in range(1, 11):
    record = router.route(qid)
    print(f"{record['query_id']:>8}{record['decision']:>8}{record['edge_conf']:>10.3f}"
          f"{record['final_pred']:>10}{record['final_conf']:>10.3f}{record['latency_ms']:>9}")

print('\n--- 路由统计分析 ---')
stats = router.analyze()
for k, v in stats.items():
    print(f'  {k}: {v}')

print('\n--- 推测式端侧解码 ---')
print(f"{'请求ID':>8}{'草稿数':>7}{'接受':>5}{'拒绝':>5}{'接受率':>7}{'推测ms':>9}{'纯云ms':>9}{'加速比':>7}")
for qid in range(1, 6):
    spec = router.speculative_edge_decode(qid, draft_tokens=8)
    print(f"{spec['query_id']:>8}{spec['draft_tokens']:>7}{spec['accepted']:>5}{spec['rejected']:>5}"
          f"{spec['accept_rate']:>7.2f}{spec['spec_latency_ms']:>9.0f}{spec['pure_cloud_latency_ms']:>9.0f}"
          f"{spec['speedup']:>7.2f}x")

print(f'\nKey: 端云协同通过置信度路由（端侧处理高置信请求，低置信转发云端）降低平均延迟与云端负载；推测式端侧解码利用端侧小模型生成草稿、云端大模型验证，可进一步加速云端推理 1.5-2 倍')

## 📝 课后思考题

1. 在端侧部署一个 7B 模型到 8GB 内存的旗舰手机，你会如何组合量化、裁剪和推理引擎技术？请说明每一步的取舍。
2. Q4_K_M 与 Q5_K_S 在体积和质量上的差异如何？什么场景下应该选择 Q5_K_S 而非 Q4_K_M？
3. 端云协同中的置信度路由阈值如何设定？阈值过高或过低分别会带来什么问题？
4. 推测式端侧解码的加速比受哪些因素影响？如果端侧小模型与云端大模型分布差异很大，会发生什么？